# Spatial Joins Exercises

Here\'s a reminder of some of the functions we have seen. Hint: they
should be useful for the exercises!

-   `sum(expression)`: aggregate to
    return a sum for a set of records
-   `count(expression)`: aggregate to
    return the size of a set of records
-   `ST_Area(geometry)` returns the
    area of the polygons
-   `ST_AsText(geometry)` returns WKT `text`
-   `ST_Contains(geometry A, geometry B)` returns the true if geometry A contains geometry B
-   `ST_Distance(geometry A, geometry B)` returns the minimum distance between geometry A and
    geometry B
-   `ST_DWithin(geometry A, geometry B, radius)` returns the true if geometry A is radius distance or less from geometry B
-   `ST_GeomFromText(text)` returns `geometry`
-   `ST_Intersects(geometry A, geometry B)` returns the true if geometry A intersects geometry B
-   `ST_Length(linestring)` returns the length of the linestring
-   `ST_Touches(geometry A, geometry B)` returns the true if the boundary of geometry A touches geometry B
-   `ST_Within(geometry A, geometry B)` returns the true if geometry A is within geometry B


Uncomment and run the following cell to install the required packages.


In [1]:
%pip install duckdb leafmap lonboard
import duckdb
import leafmap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.8/662.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 10.1 MB

Download the [nyc_data.zip](https://github.com/opengeos/data/raw/main/duckdb/nyc_data.zip) dataset using leafmap. The zip file contains the following datasets. Create a new DuckDB database and import the datasets into the database. Each dataset should be imported into a separate table.

- nyc_census_blocks
- nyc_homicides
- nyc_neighborhoods
- nyc_streets
- nyc_subway_stations

1. **What subway station is in \'Little Italy\'? What subway route is it on?**

In [2]:
url = "https://storage.googleapis.com/qm2/CASA0025/nyc_data.db.zip"
leafmap.download_file(url, unzip=True)

Downloading...
From: https://storage.googleapis.com/qm2/CASA0025/nyc_data.db.zip
To: /content/nyc_data.db.zip
100%|██████████| 8.60M/8.60M [00:00<00:00, 94.6MB/s]


Extracting files...


'/content/nyc_data.db.zip'

In [3]:
con = duckdb.connect('nyc_data.db')

In [4]:
con.install_extension('spatial')
con.load_extension('spatial')

In [5]:
con.sql("SHOW TABLES;")

┌─────────────────────┐
│        name         │
│       varchar       │
├─────────────────────┤
│ nyc_census_blocks   │
│ nyc_homicides       │
│ nyc_neighborhoods   │
│ nyc_streets         │
│ nyc_subway_stations │
└─────────────────────┘

In [17]:
con.sql("""
SELECT s.NAME
FROM nyc_subway_stations AS s
JOIN nyc_neighborhoods AS c
  ON ST_DWithin(s.geom, c.geom, 0)
  WHERE c.NAME = 'Little Italy'

""")

┌───────────┐
│   NAME    │
│  varchar  │
├───────────┤
│ Spring St │
└───────────┘

In [18]:
con.sql("""
SELECT * FROM nyc_subway_stations LIMIT 10;
""")

┌──────────┬────────┬──────────────┬─────────────────┬─────────────────┬─────────────────────────────────────────┬───────────────────────────────────┬───────────┬─────────┬─────────┬───────────┬──────────────┬─────────┬─────────┬─────────────────────────────────────────────┐
│ OBJECTID │   ID   │     NAME     │    ALT_NAME     │    CROSS_ST     │                LONG_NAME                │               LABEL               │  BOROUGH  │ NGHBHD  │ ROUTES  │ TRANSFERS │    COLOR     │ EXPRESS │ CLOSED  │                    geom                     │
│  double  │ double │   varchar    │     varchar     │     varchar     │                 varchar                 │              varchar              │  varchar  │ varchar │ varchar │  varchar  │   varchar    │ varchar │ varchar │                  geometry                   │
├──────────┼────────┼──────────────┼─────────────────┼─────────────────┼─────────────────────────────────────────┼───────────────────────────────────┼───────────┼─────────┼

2. **What are all the neighborhoods served by the 6-train?** (Hint: The `routes` column in the `nyc_subway_stations` table has values like \'B,D,6,V\' and \'C,6\')


In [21]:
con.sql("""
SELECT DISTINCT c.NAME
FROM nyc_subway_stations AS s
JOIN nyc_neighborhoods AS c
  ON ST_DWithin(s.geom, c.geom, 0)
  WHERE s.ROUTES like '%6%'

""")

┌────────────────────┐
│        NAME        │
│      varchar       │
├────────────────────┤
│ Gramercy           │
│ Murray Hill        │
│ Soundview          │
│ Upper East Side    │
│ Financial District │
│ Hunts Point        │
│ East Harlem        │
│ Little Italy       │
│ Yorkville          │
│ Chinatown          │
│ Mott Haven         │
│ Greenwich Village  │
│ Midtown            │
│ Parkchester        │
│ South Bronx        │
├────────────────────┤
│      15 rows       │
└────────────────────┘

3. **After 9/11, the \'Battery Park\' neighborhood was off limits for several days. How many people had to be evacuated?**

In [72]:
con.sql("""
SELECT  SUM(cb.popn_total)
FROM nyc_neighborhoods AS c
JOIN nyc_census_blocks AS cb
  ON ST_contains(cb.geom, c.geom, 0)
  WHERE c.NAME = 'Battery Park'

""")

BinderException: Binder Error: No function matches the given name and argument types 'ST_Contains(GEOMETRY, GEOMETRY, INTEGER_LITERAL)'. You might need to add explicit type casts.
	Candidate functions:
	ST_Contains(POLYGON_2D, POINT_2D) -> BOOLEAN
	ST_Contains(GEOMETRY, GEOMETRY) -> BOOLEAN


In [ ]:
con.sql("""
SELECT  round SUM(cb.popn_total)
FROM nyc_neighborhoods AS c
JOIN nyc_census_blocks AS cb
  ON ST_DWithin(cb.geom, c.geom, 0)
  WHERE c.NAME = 'Battery Park'

""")

In [36]:
con.sql("""
SELECT sum(st_area(st_intersection(c.geom, cb.geom)) / st_area(cb.geom) * cb.popn_total)
FROM nyc_neighborhoods AS c
JOIN nyc_census_blocks AS cb
  ON ST_Intersects(cb.geom, c.geom)
  WHERE c.NAME = 'Battery Park';

""")

┌───────────────────────────────────────────────────────────────────────────────────────┐
│ sum(((st_area(st_intersection(c.geom, cb.geom)) / st_area(cb.geom)) * cb.popn_total)) │
│                                        double                                         │
├───────────────────────────────────────────────────────────────────────────────────────┤
│                                                                     11917.15724605854 │
└───────────────────────────────────────────────────────────────────────────────────────┘

4. **What neighborhood has the highest population density (persons/km2)?**


In [69]:
con.sql("""
SELECT  c.NAME,
1000000*SUM(cb.popn_total)/ mean(ST_Area(c.geom)) AS DENSITY
FROM nyc_census_blocks AS cb
JOIN nyc_neighborhoods AS c
  ON ST_intersects(c.geom, cb.geom)
  GROUP BY c.NAME
ORDER BY DENSITY DESC LIMIT 1;


""")

┌───────────────────┬───────────────────┐
│       NAME        │      DENSITY      │
│      varchar      │      double       │
├───────────────────┼───────────────────┤
│ North Sutton Area │ 68435.13283772676 │
└───────────────────┴───────────────────┘

When you're finished, you can check your answers [here](https://postgis.net/workshops/postgis-intro/joins_exercises.html).

# Ship-to-Ship Transfer Detection

Now for a less structured exercise. We're going to look at ship-to-ship transfers. The idea is that two ships meet up in the middle of the ocean, and one ship transfers cargo to the other. This is a common way to avoid sanctions, and is often used to transfer oil from sanctioned countries to other countries. We're going to look at a few different ways to detect these transfers using AIS data.

In [61]:
pip install duckdb duckdb-engine jupysql

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.4/192.4 kB 10.2 MB/s eta 0:00:00


In [62]:
import duckdb
import pandas as pd

# Import jupysql Jupyter extension to create SQL cells
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False
%sql duckdb:///:memory:

In [70]:
%%sql
INSTALL httpfs;
LOAD httpfs;
INSTALL spatial;
LOAD spatial;

,Success


## Step 1

Create a spatial database using the following AIS data:

https://storage.googleapis.com/qm2/casa0025_ships.csv

Each row in this dataset is an AIS 'ping' indicating the position of a ship at a particular date/time, alongside vessel-level characteristics.

It contains the following columns:
* `vesselid`: A unique numerical identifier for each ship, like a license plate
* `vessel_name`: The ship's name
* `vsl_descr`: The ship's type
* `dwt`: The ship's Deadweight Tonnage (how many tons it can carry)
* `v_length`: The ship's length in meters
* `draught`: How many meters deep the ship is draughting (how low it sits in the water). Effectively indicates how much cargo the ship is carrying
* `sog`: Speed over Ground (in knots)
* `date`: A timestamp for the AIS signal
* `lat`: The latitude of the AIS signal (EPSG:4326)
* `lon`: The longitude of the AIS signal (EPSG:4326)

Create a table called 'ais' where each row is a different AIS ping, with no superfluous information. Construct a geometry column.

Create a second table called 'vinfo' which contains vessel-level information with no superfluous information.

You can set a spatial index on each of these tables as follows:

`CREATE INDEX index_name ON table_name USING RTREE(geom);`

In [73]:
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

In [74]:
%sql duckdb:///:memory:
# %sql duckdb:///path/to/file.db

In [75]:
%%sql

SELECT * FROM duckdb_extensions();

,extension_name,loaded,installed,install_path,description,aliases,extension_version,install_mode,installed_from
0,autocomplete,False,False,,Adds support for autocomplete in the shell,[],,NOT_INSTALLED,
1,aws,False,False,,Provides features that depend on the AWS SDK,[],,NOT_INSTALLED,
2,azure,False,False,,Adds a filesystem abstraction for Azure blob s...,[],,NOT_INSTALLED,
3,core_functions,True,True,(BUILT-IN),Core function library,[],v1.4.4,STATICALLY_LINKED,
4,delta,False,False,,Adds support for Delta Lake,[],,NOT_INSTALLED,
5,ducklake,False,False,,"Adds support for DuckLake, SQL as a Lakehouse ...",[],,NOT_INSTALLED,
6,encodings,False,False,,All unicode encodings to UTF-8,[],,NOT_INSTALLED,
7,excel,False,False,,Adds support for Excel-like format strings,[],,NOT_INSTALLED,
8,fts,False,False,,Adds support for Full-Text Search Indexes,[],,NOT_INSTALLED,
9,httpfs,True,True,/root/.duckdb/extensions/v1.4.4/linux_amd64/ht...,Adds support for reading and writing files ove...,"[http, https, s3]",13f8a81,REPOSITORY,core


In [76]:
%%sql

INSTALL httpfs;
LOAD httpfs;

,Success


In [77]:
%%sql

SELECT * FROM 'https://storage.googleapis.com/qm2/casa0025_ships.csv';

,vesselid,vessel_name,vsl_descr,dwt,v_length,draught,sog,date,lat,lon,geom
0,350053,30 Let Pobedy,general cargo,5150.0,NaN,3.5,5.2,2022-07-25 02:53:29,45.151777,36.513327,POINT (36.5133266666667 45.1517766666667)
1,350053,30 Let Pobedy,general cargo,5150.0,NaN,3.5,0.7,2022-07-25 03:09:37,45.146487,36.520780,POINT (36.52078 45.1464866666667)
2,350053,30 Let Pobedy,general cargo,5150.0,NaN,3.5,0.7,2022-07-25 03:13:58,45.146218,36.521965,POINT (36.521965 45.1462183333333)
3,350053,30 Let Pobedy,general cargo,5150.0,NaN,3.5,0.1,2022-07-25 04:16:06,45.145058,36.522020,POINT (36.52202 45.1450583333333)
4,350053,30 Let Pobedy,general cargo,5150.0,NaN,3.5,0.0,2022-07-25 05:20:17,45.144933,36.521848,POINT (36.5218483333333 45.1449333333333)
...,...,...,...,...,...,...,...,...,...,...,...
101323,217531,Zubeyde,roll on roll off with container capacity,5000.0,113.0,4.5,0.1,2022-08-10 14:16:47,45.091987,36.522157,POINT (36.5221566666667 45.0919866666667)
101324,217531,Zubeyde,roll on roll off with container capacity,5000.0,113.0,4.5,0.1,2022-08-10 14:43:48,45.091643,36.522213,POINT (36.5222133333333 45.0916433333333)
101325,217531,Zubeyde,roll on roll off with container capacity,5000.0,113.0,4.5,5.8,2022-08-10 15:04:28,45.100457,36.519397,POINT (36.5193966666667 45.1004566666667)
101326,217531,Zubeyde,roll on roll off with container capacity,5000.0,113.0,4.5,8.3,2022-08-23 06:06:51,45.087527,36.506987,POINT (36.5069866666667 45.0875266666667)


## Step 2

Use a spatial join to identify ship-to-ship transfers in this dataset.
Two ships are considered to be conducting a ship to ship transfer IF:

* They are within 500 meters of each other
* For more than two hours
* And their speed is lower than 1 knot

Some things to consider: make sure you're not joining ships with themselves. Try working with subsets of the data first while you try different things out.